In [ ]:
# %% 셀 1: 데이터 로드 (Model 2 - Box set, identity-free)
import os, json, random, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import polars as pl
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed
from scipy.optimize import linear_sum_assignment

POS_DIR = "./data/8_telop_position"
STT_DIR = "./data/4_stt_results"
EMB_PATH = "./data/8_text_embeddings.pt"
GRID_W = 80
GRID_H = 80
EVAL_PER_CHANNEL = 5
SEED = 42
NUM_WORKERS = 32
FPS = 10
MAX_FRAMES = 2000
MAX_ACTIVE_PER_FRAME = 150
MAX_TEXT_LEN = 200
SCALAR_DIM = 9
TIME_DIM = 3

text2emb = torch.load(EMB_PATH, weights_only=True)
EMB_DIM = next(iter(text2emb.values())).shape[0]
ZERO_EMB = torch.zeros(EMB_DIM)
print(f"✅ 임베딩 로드: {len(text2emb):,}개  |  dim={EMB_DIM}")


def stt_frames_to_segments(df_stt):
    rows = df_stt.sort("frame_num").to_dicts()
    segments = []
    cur_text = None
    cur_start_frame = None
    prev_frame = None
    for r in rows:
        t = r["stt_text"]
        f = int(r["frame_num"])
        if t != cur_text:
            if cur_text is not None and cur_text != "":
                segments.append({
                    "start": cur_start_frame / FPS,
                    "end": (prev_frame + 1) / FPS,
                    "text": cur_text.strip(),
                })
            cur_text = t
            cur_start_frame = f
        prev_frame = f
    if cur_text is not None and cur_text != "":
        segments.append({
            "start": cur_start_frame / FPS,
            "end": (prev_frame + 1) / FPS,
            "text": cur_text.strip(),
        })
    return segments


def load_one(args):
    channel, path = args
    with open(path, "r") as f:
        data = json.load(f)
    instances = data.get("instances", [])
    duration = data.get("duration", 0.1)
    if instances:
        duration = max(duration, max(inst["end_sec"] for inst in instances))
    video_name = data.get("video", "")
    file_id = os.path.basename(path)[:-5]
    inst_list = []
    for inst in instances:
        gx = int(np.clip(round(inst["grid_x"]), 0, GRID_W - 1))
        gy = int(np.clip(round(inst["grid_y"]), 0, GRID_H - 1))
        gw = int(np.clip(round(inst["grid_w"]), 1, GRID_W))
        gh = int(np.clip(round(inst["grid_h"]), 1, GRID_H))
        inst_list.append({
            "text": inst["text"],
            "text_len": len(inst["text"]),
            "start": inst["start_sec"],
            "end": inst["end_sec"],
            "x": gx, "y": gy, "w": gw, "h": gh,
        })
    stt_path = os.path.join(STT_DIR, channel, file_id + ".parquet")
    stt_segments = []
    if os.path.exists(stt_path):
        try:
            df_stt = pl.read_parquet(stt_path, glob=False)
            stt_segments = stt_frames_to_segments(df_stt)
        except:
            pass
    return {
        "channel": channel,
        "video_name": video_name,
        "file_id": file_id,
        "instances": inst_list,
        "stt_segments": stt_segments,
        "duration": duration,
    }


json_paths = []
for channel in sorted(os.listdir(POS_DIR)):
    ch_dir = os.path.join(POS_DIR, channel)
    if not os.path.isdir(ch_dir):
        continue
    for fname in sorted(os.listdir(ch_dir)):
        if fname.endswith(".json"):
            json_paths.append((channel, os.path.join(ch_dir, fname)))

samples = []
channel_set = set()
with ProcessPoolExecutor(max_workers=NUM_WORKERS) as pool:
    futures = {pool.submit(load_one, args): args for args in json_paths}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="로드"):
        result = fut.result()
        channel_set.add(result["channel"])
        samples.append(result)

channels = sorted(channel_set)
channel2id = {ch: i for i, ch in enumerate(channels)}

rng = random.Random(SEED)
by_channel = {}
for s in samples:
    by_channel.setdefault(s["channel"], []).append(s)

train_samples = []
eval_samples = []
for ch, ch_samples in by_channel.items():
    ch_samples.sort(key=lambda s: s["file_id"])
    rng.shuffle(ch_samples)
    n_eval = min(EVAL_PER_CHANNEL, len(ch_samples))
    eval_samples.extend(ch_samples[:n_eval])
    train_samples.extend(ch_samples[n_eval:])

train_samples = [s for s in train_samples if len(s["instances"]) > 0]
eval_samples_with = [s for s in eval_samples if len(s["instances"]) > 0]

print(f"✅ 채널: {len(channels)}개")
print(f"   train: {len(train_samples):,}  eval: {len(eval_samples_with):,}")

In [ ]:
# %% 셀 1b: 채널 서브샘플링 (debug)
rng_ch = random.Random(42)
sampled_channels = rng_ch.sample(channels, 67)
sampled_set = set(sampled_channels)

train_samples = [s for s in train_samples if s["channel"] in sampled_set]
eval_samples = [s for s in eval_samples if s["channel"] in sampled_set]
channels = sampled_channels
channel2id = {ch: i for i, ch in enumerate(channels)}

eval_samples_with = [s for s in eval_samples if len(s["instances"]) > 0]
train_samples = [s for s in train_samples if len(s["instances"]) > 0]

print(f"\n🔬 샘플링: {len(channels)}개 채널")
print(f"   train: {len(train_samples):,}  |  eval: {len(eval_samples):,}")

In [ ]:
# %% 셀 2: Dataset + DataLoader (segment 단위 + CenterNet GT — center heatmap, geometry, pos_mask)
from torch.utils.data import Dataset, DataLoader

BATCH_SIZE = 8
NUM_WORKERS_DL = 8

CN_GAUSS_RATIO = 0.5    # Gaussian sigma = ratio × sqrt(w*h) (in pixel units)
CN_GAUSS_MIN_SIGMA = 1.0
CN_GAUSS_TRUNC = 3.0    # truncate at 3 sigma
LOG_W_FLOOR = 1.0 / (GRID_W * 2)   # 안전 floor for log(w_norm)
LOG_H_FLOOR = 1.0 / (GRID_H * 2)


def _draw_gaussian(heatmap, cx, cy, sigma):
    """In-place: heatmap[y,x] = max(heatmap[y,x], exp(-((x-cx)^2+(y-cy)^2)/(2*sigma^2)))"""
    H, W = heatmap.shape
    radius = int(np.ceil(CN_GAUSS_TRUNC * sigma))
    x0, x1 = max(0, cx - radius), min(W, cx + radius + 1)
    y0, y1 = max(0, cy - radius), min(H, cy + radius + 1)
    if x0 >= x1 or y0 >= y1:
        return
    yy, xx = np.meshgrid(np.arange(y0, y1), np.arange(x0, x1), indexing='ij')
    g = np.exp(-((xx - cx) ** 2 + (yy - cy) ** 2) / (2.0 * sigma * sigma))
    np.maximum(heatmap[y0:y1, x0:x1], g, out=heatmap[y0:y1, x0:x1])


class BoxSetSegmentDataset(Dataset):
    """프레임 대신 segment 단위로 반환. GT: cxcywh boxes + CenterNet targets."""
    def __init__(self, samples, channel2id, text2emb):
        self.samples = [s for s in samples if len(s["instances"]) > 0]
        self.channel2id = channel2id
        self.text2emb = text2emb

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        channel_id = self.channel2id[s["channel"]]
        instances = s["instances"]
        duration = max(s["duration"], 0.1)
        n_frames = max(1, min(int(duration * FPS) + 1, MAX_FRAMES))
        n_inst = len(instances)

        inst_starts = np.array([inst["start"] for inst in instances])
        inst_ends   = np.array([inst["end"]   for inst in instances])
        inst_tlens  = np.array([inst["text_len"] for inst in instances])
        inst_cx = np.array([inst["x"] for inst in instances])
        inst_cy = np.array([inst["y"] for inst in instances])
        inst_w  = np.array([inst["w"] for inst in instances])
        inst_h  = np.array([inst["h"] for inst in instances])

        inst_x0 = np.maximum(0, inst_cx - inst_w // 2)
        inst_y0 = np.maximum(0, inst_cy - inst_h // 2)
        inst_x1 = np.minimum(GRID_W, inst_x0 + inst_w)
        inst_y1 = np.minimum(GRID_H, inst_y0 + inst_h)

        # 임베딩
        channel_emb = F.normalize(self.text2emb.get(s["channel"], ZERO_EMB), dim=-1)
        video_emb   = F.normalize(self.text2emb.get(s["video_name"], ZERO_EMB), dim=-1)
        inst_embs   = torch.stack([
            self.text2emb.get(inst["text"], ZERO_EMB) for inst in instances])
        inst_embs = F.normalize(inst_embs, dim=-1)

        inst_diff_ch  = inst_embs - channel_emb.unsqueeze(0)
        inst_diff_vid = inst_embs - video_emb.unsqueeze(0)
        inst_diff_stt = torch.zeros(n_inst, EMB_DIM)

        ch_sims  = F.cosine_similarity(inst_embs, channel_emb.unsqueeze(0), dim=-1).numpy()
        vid_sims = F.cosine_similarity(inst_embs, video_emb.unsqueeze(0), dim=-1).numpy()

        stt_sims = np.zeros(n_inst, dtype=np.float32)
        has_stts = np.zeros(n_inst, dtype=np.float32)
        stt_segments = s["stt_segments"]
        if len(stt_segments) > 0:
            stt_starts = np.array([seg["start"] for seg in stt_segments])
            stt_ends   = np.array([seg["end"]   for seg in stt_segments])
            stt_embs_raw = torch.stack([
                self.text2emb.get(seg["text"], ZERO_EMB) for seg in stt_segments])
            stt_embs = F.normalize(stt_embs_raw, dim=-1)
            for i in range(n_inst):
                mid = (inst_starts[i] + inst_ends[i]) / 2
                stt_active = (stt_starts <= mid) & (stt_ends > mid)
                stt_active_idx = np.where(stt_active)[0]
                if len(stt_active_idx) > 0:
                    inst_diff_stt[i] = inst_embs[i] - stt_embs[stt_active_idx[0]]
                    stt_sims[i] = F.cosine_similarity(
                        inst_embs[i].unsqueeze(0), stt_embs[stt_active_idx[0]].unsqueeze(0)).item()
                    has_stts[i] = 1.0

        times = np.arange(n_frames, dtype=np.float32) / FPS
        active_matrix = (
            (inst_starts[None, :] <= times[:, None] + 0.05) &
            (inst_ends[None, :]   >  times[:, None]))

        co_active_per_frame = active_matrix.sum(axis=1)
        inst_avg_coactive = np.zeros(n_inst, dtype=np.float32)
        for i in range(n_inst):
            frames_i = active_matrix[:, i]
            if frames_i.any():
                inst_avg_coactive[i] = co_active_per_frame[frames_i].mean()
        inst_avg_coactive = np.log1p(inst_avg_coactive) / np.log1p(20.0)

        inst_scalars = np.zeros((n_inst, SCALAR_DIM), dtype=np.float32)
        inst_scalars[:, 0] = inst_tlens / MAX_TEXT_LEN
        inst_scalars[:, 1] = ch_sims
        inst_scalars[:, 2] = vid_sims
        inst_scalars[:, 3] = stt_sims
        inst_scalars[:, 4] = has_stts
        inst_scalars[:, 5] = inst_starts / max(duration, 0.1)
        inst_scalars[:, 6] = inst_ends / max(duration, 0.1)
        inst_scalars[:, 7] = (inst_ends - inst_starts) / max(duration, 0.1)
        inst_scalars[:, 8] = inst_avg_coactive

        if n_frames > 1:
            diff = (active_matrix[1:] != active_matrix[:-1]).any(axis=-1)
            boundaries = np.concatenate([[0], np.where(diff)[0] + 1, [n_frames]])
        else:
            boundaries = np.array([0, n_frames])

        n_segs = len(boundaries) - 1
        seg_active_masks = np.zeros((n_segs, n_inst), dtype=np.bool_)
        seg_merged_masks = np.zeros((n_segs, GRID_H, GRID_W), dtype=np.float32)
        seg_gt_boxes = np.zeros((n_segs, MAX_ACTIVE_PER_FRAME, 4), dtype=np.float32)
        seg_gt_box_masks = np.zeros((n_segs, MAX_ACTIVE_PER_FRAME), dtype=np.bool_)
        seg_time_feats = np.zeros((n_segs, TIME_DIM), dtype=np.float32)
        seg_lengths = np.zeros(n_segs, dtype=np.int64)

        # ── CenterNet GT ──
        seg_center_target = np.zeros((n_segs, GRID_H, GRID_W), dtype=np.float32)
        seg_geo_target    = np.zeros((n_segs, 4, GRID_H, GRID_W), dtype=np.float32)
        seg_pos_mask      = np.zeros((n_segs, GRID_H, GRID_W), dtype=np.bool_)

        for s_i in range(n_segs):
            f_start = boundaries[s_i]
            f_end = boundaries[s_i + 1]
            f_rep = (f_start + f_end) // 2
            seg_lengths[s_i] = f_end - f_start

            active = active_matrix[f_rep]
            seg_active_masks[s_i] = active

            for j in np.where(active)[0]:
                x0, y0, x1, y1 = int(inst_x0[j]), int(inst_y0[j]), int(inst_x1[j]), int(inst_y1[j])
                if x1 > x0 and y1 > y0:
                    seg_merged_masks[s_i, y0:y1, x0:x1] = 1.0

            active_idx = np.where(active)[0]
            if len(active_idx) > 0:
                # 큰 박스 먼저 처리 → center pixel 충돌 시 큰 박스 보존
                areas = (inst_x1[active_idx] - inst_x0[active_idx]) * \
                        (inst_y1[active_idx] - inst_y0[active_idx])
                order_area = np.argsort(-areas)
                sorted_by_area = active_idx[order_area]

                for j in sorted_by_area:
                    bx0, by0 = int(inst_x0[j]), int(inst_y0[j])
                    bx1, by1 = int(inst_x1[j]), int(inst_y1[j])
                    if bx1 <= bx0 or by1 <= by0:
                        continue
                    cx_f = (bx0 + bx1) / 2.0
                    cy_f = (by0 + by1) / 2.0
                    w_f  = bx1 - bx0
                    h_f  = by1 - by0
                    cx_int = int(np.clip(round(cx_f), 0, GRID_W - 1))
                    cy_int = int(np.clip(round(cy_f), 0, GRID_H - 1))

                    sigma = max(CN_GAUSS_MIN_SIGMA, CN_GAUSS_RATIO * np.sqrt(w_f * h_f))
                    _draw_gaussian(seg_center_target[s_i], cx_int, cy_int, sigma)

                    if not seg_pos_mask[s_i, cy_int, cx_int]:
                        # CenterNet 표준 인코딩: (offset_x, offset_y, log_w_norm, log_h_norm)
                        #   offset = sub-pixel residual in [-0.5, 0.5]
                        #   log_size = scale-invariant, raw output (no activation)
                        w_norm = max(w_f / GRID_W, LOG_W_FLOOR)
                        h_norm = max(h_f / GRID_H, LOG_H_FLOOR)
                        seg_geo_target[s_i, 0, cy_int, cx_int] = cx_f - cx_int
                        seg_geo_target[s_i, 1, cy_int, cx_int] = cy_f - cy_int
                        seg_geo_target[s_i, 2, cy_int, cx_int] = np.log(w_norm)
                        seg_geo_target[s_i, 3, cy_int, cx_int] = np.log(h_norm)
                        seg_pos_mask[s_i, cy_int, cx_int] = True

                # gt_boxes (text_len 내림차순) — eval metric용
                sorted_order = np.argsort(inst_tlens[active_idx])[::-1][:MAX_ACTIVE_PER_FRAME]
                sorted_idx = active_idx[sorted_order]
                for k_i, j in enumerate(sorted_idx):
                    cx = (inst_x0[j] + inst_x1[j]) / 2 / GRID_W
                    cy = (inst_y0[j] + inst_y1[j]) / 2 / GRID_H
                    w = (inst_x1[j] - inst_x0[j]) / GRID_W
                    h = (inst_y1[j] - inst_y0[j]) / GRID_H
                    seg_gt_boxes[s_i, k_i] = [cx, cy, w, h]
                    seg_gt_box_masks[s_i, k_i] = True

            t_norm = times[f_rep] / max(duration, 1.0)
            seg_time_feats[s_i] = [t_norm, np.sin(2*np.pi*t_norm), np.cos(2*np.pi*t_norm)]

        return {
            "channel_id":        torch.tensor(channel_id, dtype=torch.long),
            "inst_diff_ch":      inst_diff_ch,
            "inst_diff_vid":     inst_diff_vid,
            "inst_diff_stt":     inst_diff_stt,
            "inst_scalars":      torch.from_numpy(inst_scalars),
            "n_inst":            n_inst,
            "seg_active_mask":   torch.from_numpy(seg_active_masks),
            "seg_merged_mask":   torch.from_numpy(seg_merged_masks),
            "seg_gt_boxes":      torch.from_numpy(seg_gt_boxes),
            "seg_gt_box_mask":   torch.from_numpy(seg_gt_box_masks),
            "seg_time_feats":    torch.from_numpy(seg_time_feats),
            "seg_length":        torch.from_numpy(seg_lengths),
            "seg_center_target": torch.from_numpy(seg_center_target),
            "seg_geo_target":    torch.from_numpy(seg_geo_target),
            "seg_pos_mask":      torch.from_numpy(seg_pos_mask),
            "n_segments":        n_segs,
        }


def collate_fn(batch):
    B = len(batch)
    max_inst = max(b["inst_diff_ch"].shape[0] for b in batch)

    channel_ids = torch.stack([b["channel_id"] for b in batch])
    inst_diff_ch  = torch.zeros(B, max_inst, EMB_DIM)
    inst_diff_vid = torch.zeros(B, max_inst, EMB_DIM)
    inst_diff_stt = torch.zeros(B, max_inst, EMB_DIM)
    inst_scalars  = torch.zeros(B, max_inst, SCALAR_DIM)

    for i, b in enumerate(batch):
        n_inst = b["inst_diff_ch"].shape[0]
        inst_diff_ch[i, :n_inst]  = b["inst_diff_ch"]
        inst_diff_vid[i, :n_inst] = b["inst_diff_vid"]
        inst_diff_stt[i, :n_inst] = b["inst_diff_stt"]
        inst_scalars[i, :n_inst]  = b["inst_scalars"]

    all_active = []; all_merged = []; all_gt = []; all_gtm = []
    all_time = []; all_seg_len = []; all_vidx = []
    all_center = []; all_geo = []; all_pos = []

    for i, b in enumerate(batch):
        n_inst = b["n_inst"]
        n_seg = b["n_segments"]
        ap = torch.zeros(n_seg, max_inst, dtype=torch.bool)
        ap[:, :n_inst] = b["seg_active_mask"]
        all_active.append(ap)
        all_merged.append(b["seg_merged_mask"])
        all_gt.append(b["seg_gt_boxes"])
        all_gtm.append(b["seg_gt_box_mask"])
        all_time.append(b["seg_time_feats"])
        all_seg_len.append(b["seg_length"])
        all_vidx.append(torch.full((n_seg,), i, dtype=torch.long))
        all_center.append(b["seg_center_target"])
        all_geo.append(b["seg_geo_target"])
        all_pos.append(b["seg_pos_mask"])

    return {
        "channel_ids":        channel_ids,
        "inst_diff_ch":       inst_diff_ch,
        "inst_diff_vid":      inst_diff_vid,
        "inst_diff_stt":      inst_diff_stt,
        "inst_scalars":       inst_scalars,
        "seg_active_mask":    torch.cat(all_active, dim=0),
        "seg_merged_mask":    torch.cat(all_merged, dim=0),
        "seg_gt_boxes":       torch.cat(all_gt, dim=0),
        "seg_gt_box_mask":    torch.cat(all_gtm, dim=0),
        "seg_time_feats":     torch.cat(all_time, dim=0),
        "seg_length":         torch.cat(all_seg_len, dim=0),
        "seg_video_idx":      torch.cat(all_vidx, dim=0),
        "seg_center_target":  torch.cat(all_center, dim=0),
        "seg_geo_target":     torch.cat(all_geo, dim=0),
        "seg_pos_mask":       torch.cat(all_pos, dim=0),
    }


train_ds = BoxSetSegmentDataset(train_samples, channel2id, text2emb)
eval_ds  = BoxSetSegmentDataset(eval_samples, channel2id, text2emb)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS_DL, pin_memory=True,
    collate_fn=collate_fn, drop_last=True,
    persistent_workers=True,
)
eval_loader = DataLoader(
    eval_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS_DL, pin_memory=True,
    collate_fn=collate_fn,
    persistent_workers=True,
)

print(f"✅ Dataset (segment + CenterNet GT): train={len(train_ds):,}  eval={len(eval_ds):,}")

batch = next(iter(train_loader))
print(f"\n✅ 배치 확인 (segment-flattened)")
for k, v in batch.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: {tuple(v.shape)}  {v.dtype}")
print(f"\n  S_total: {batch['seg_video_idx'].shape[0]}")
print(f"  pos pixels per seg: mean={batch['seg_pos_mask'].sum(dim=[1,2]).float().mean().item():.1f}  "
      f"max={int(batch['seg_pos_mask'].sum(dim=[1,2]).max().item())}")
print(f"  center_target max: {batch['seg_center_target'].max().item():.3f}")
# geo_target sanity (offset, offset, log_w, log_h)
_pm = batch['seg_pos_mask']
_pos_idx = _pm.nonzero(as_tuple=True)
if _pos_idx[0].numel() > 0:
    _g = batch['seg_geo_target'][_pos_idx[0], :, _pos_idx[1], _pos_idx[2]]
    print(f"  geo_target at pos: offset_x range=[{_g[:,0].min():.3f}, {_g[:,0].max():.3f}]  "
          f"log_w range=[{_g[:,2].min():.3f}, {_g[:,2].max():.3f}]")


In [ ]:
# %% 셀 3: 모델 정의 (CenterNet 스타일 dense prediction — center heatmap + geometry map)
D_MODEL = 256
N_HEADS = 8
N_LAYERS_INST = 4
N_BACKBONE = 4          # backbone conv 레이어 수
D_FF = 512
DROPOUT = 0.1
SPATIAL_STRIDE = 1
SPATIAL_H = GRID_H // SPATIAL_STRIDE
SPATIAL_W = GRID_W // SPATIAL_STRIDE


class CenterNetBlock(nn.Module):
    """Conv block with optional FiLM conditioning."""
    def __init__(self, d_model, kernel=3):
        super().__init__()
        self.conv1 = nn.Conv2d(d_model, d_model, kernel, padding=kernel // 2)
        self.conv2 = nn.Conv2d(d_model, d_model, kernel, padding=kernel // 2)
        self.norm1 = nn.GroupNorm(8, d_model)
        self.norm2 = nn.GroupNorm(8, d_model)
        self.act = nn.GELU()

    def forward(self, x, gain=None, bias=None):
        # First conv + FiLM (optional) + GN + GELU
        out = self.conv1(x)
        if gain is not None:
            out = out * (1 + gain) + bias
        out = self.act(self.norm1(out))
        out = self.act(self.norm2(self.conv2(out)))
        return out + x      # residual


class CenterNetModel(nn.Module):
    def __init__(self, n_channels, emb_dim=EMB_DIM, d_model=D_MODEL,
                 n_heads=N_HEADS, d_ff=D_FF, dropout=DROPOUT):
        super().__init__()
        self.d_model = d_model

        # ── 1. Instance encoder ──
        self.ch_proj = nn.Sequential(nn.Linear(emb_dim + 1, d_model // 2), nn.GELU())
        self.vid_proj = nn.Sequential(nn.Linear(emb_dim + 1, d_model // 2), nn.GELU())
        self.stt_proj = nn.Sequential(nn.Linear(emb_dim + 2, d_model // 2), nn.GELU())
        self.len_proj = nn.Sequential(nn.Linear(5, d_model // 2), nn.GELU())
        self.slot_combine = nn.Sequential(
            nn.Linear(d_model * 2, d_model), nn.GELU(),
            nn.Linear(d_model, d_model))
        inst_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, activation="gelu")
        self.inst_self_attn = nn.TransformerEncoder(
            inst_layer, num_layers=N_LAYERS_INST, enable_nested_tensor=False)

        self.channel_emb = nn.Embedding(n_channels, d_model)

        # ── 2. Input heatmap → spatial feature ──
        self.input_conv = nn.Sequential(
            nn.Conv2d(1, d_model // 2, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(d_model // 2, d_model, 3, padding=1),
        )
        self.spatial_pos_emb = nn.Parameter(torch.zeros(1, d_model, SPATIAL_H, SPATIAL_W))
        nn.init.normal_(self.spatial_pos_emb, std=0.02)

        # ── 3. Frame context (channel/time/count + active inst pool) → FiLM (gain, bias) ──
        self.time_proj = nn.Linear(TIME_DIM, d_model)
        self.count_proj = nn.Sequential(
            nn.Linear(1, d_model // 4), nn.GELU(),
            nn.Linear(d_model // 4, d_model))
        self.ctx_combine = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(),
            nn.Linear(d_model, d_model))
        # 각 backbone block마다 별도 FiLM (gain, bias)
        self.film = nn.ModuleList([
            nn.Linear(d_model, d_model * 2) for _ in range(N_BACKBONE)
        ])

        # ── 4. Backbone (FiLM-conditioned residual conv blocks) ──
        self.backbone = nn.ModuleList([
            CenterNetBlock(d_model, kernel=3) for _ in range(N_BACKBONE)
        ])

        # ── 5. Heads ──
        self.center_head = nn.Sequential(
            nn.Conv2d(d_model, d_model // 2, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(d_model // 2, 1, 1),
        )
        # focal loss prior — 초기엔 거의 모든 픽셀이 negative
        nn.init.constant_(self.center_head[-1].bias, -2.19)   # sigmoid(−2.19) ≈ 0.1

        self.geometry_head = nn.Sequential(
            nn.Conv2d(d_model, d_model // 2, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(d_model // 2, 4, 1),
        )
        # geometry: raw 출력 (offset_x, offset_y, log_w_norm, log_h_norm)
        # — offset은 [-0.5, 0.5] 범위, log_size는 raw real number
        # bias=0 (기본): 초기 출력 (0, 0, 0, 0) → cx=픽셀, cy=픽셀, w=1, h=1 (clamp 필요)

    def encode_inst(self, channel_ids, inst_diff_ch, inst_diff_vid,
                    inst_diff_stt, inst_scalars):
        ch_input = torch.cat([inst_diff_ch, inst_scalars[..., 1:2]], dim=-1)
        vid_input = torch.cat([inst_diff_vid, inst_scalars[..., 2:3]], dim=-1)
        stt_input = torch.cat([inst_diff_stt,
                               inst_scalars[..., 3:4],
                               inst_scalars[..., 4:5]], dim=-1)
        len_input = torch.cat([
            inst_scalars[..., 0:1],
            inst_scalars[..., 5:8],
            inst_scalars[..., 8:9]], dim=-1)

        proj_ch  = self.ch_proj(ch_input)
        proj_vid = self.vid_proj(vid_input)
        proj_stt = self.stt_proj(stt_input)
        proj_len = self.len_proj(len_input)

        inst_tokens = self.slot_combine(
            torch.cat([proj_ch, proj_vid, proj_stt, proj_len], dim=-1))
        ch_emb = self.channel_emb(channel_ids).unsqueeze(1)
        inst_tokens = inst_tokens + ch_emb

        inst_mask = (inst_scalars[..., 0] != 0)
        inst_pad = ~inst_mask
        inst_tokens = self.inst_self_attn(inst_tokens, src_key_padding_mask=inst_pad)
        return inst_tokens, inst_mask

    def predict_segments(self, inst_tokens, channel_ids,
                         seg_active_mask, seg_merged_mask, seg_time_feats, seg_video_idx):
        """
        Dense prediction:
          center_logits: (S, 1, H, W) — sigmoid for objectness
          geometry:      (S, 4, H, W) — cxcywh in [0,1] (sigmoid)
        """
        S = seg_merged_mask.shape[0]
        D = self.d_model

        # Spatial features (heatmap → conv)
        h = seg_merged_mask.unsqueeze(1)                 # (S, 1, H, W)
        spatial = self.input_conv(h)                     # (S, D, H, W)
        spatial = spatial + self.spatial_pos_emb         # broadcast (1, D, H, W)

        # Active inst pool per segment
        inst_per_seg = inst_tokens[seg_video_idx]                          # (S, max_inst, D)
        active_f = seg_active_mask.float().unsqueeze(-1)                   # (S, max_inst, 1)
        n_active = active_f.sum(dim=1).clamp(min=1.0)                      # (S, 1)
        inst_pool = (inst_per_seg * active_f).sum(dim=1) / n_active        # (S, D)

        # Frame context
        ch_emb = self.channel_emb(channel_ids[seg_video_idx])              # (S, D)
        time_emb = self.time_proj(seg_time_feats)                          # (S, D)
        k_per_seg = seg_active_mask.sum(dim=-1)
        count_norm = torch.log1p(k_per_seg.float().unsqueeze(-1)) / math.log1p(20.0)
        count_emb = self.count_proj(count_norm)                            # (S, D)

        ctx = self.ctx_combine(ch_emb + time_emb + count_emb + inst_pool)  # (S, D)

        # Backbone with FiLM
        feat = spatial
        for blk, film_l in zip(self.backbone, self.film):
            params = film_l(ctx)                                           # (S, 2D)
            gain, bias = params.chunk(2, dim=-1)                           # (S, D), (S, D)
            gain = gain.unsqueeze(-1).unsqueeze(-1)                        # (S, D, 1, 1)
            bias = bias.unsqueeze(-1).unsqueeze(-1)
            feat = blk(feat, gain=gain, bias=bias)

        # Heads
        center_logits = self.center_head(feat)                             # (S, 1, H, W)
        geometry = self.geometry_head(feat)                                # (S, 4, H, W) raw
        return center_logits, geometry

    def forward(self, channel_ids, inst_diff_ch, inst_diff_vid, inst_diff_stt, inst_scalars,
                seg_active_mask, seg_merged_mask, seg_time_feats, seg_video_idx):
        inst_tokens, _ = self.encode_inst(
            channel_ids, inst_diff_ch, inst_diff_vid, inst_diff_stt, inst_scalars)
        return self.predict_segments(
            inst_tokens, channel_ids,
            seg_active_mask, seg_merged_mask, seg_time_feats, seg_video_idx)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CenterNetModel(n_channels=len(channels)).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"🖥️  Device: {device}")
print(f"📐 파라미터: {n_params:,}")
print(f"   inst self-attn {N_LAYERS_INST}L (D={D_MODEL})")
print(f"   input: heatmap (1, 80, 80) → spatial feature (D={D_MODEL})")
print(f"   ctx: ch + time + count + inst_pool → FiLM gain/bias per block")
print(f"   backbone: {N_BACKBONE}× FiLM-conditioned residual conv (3×3)")
print(f"   heads: center (sigmoid for objectness), geometry (raw: offset_xy + log_w/h_norm)")
print(f"   center_head bias init = -2.19 (focal loss prior)")


In [ ]:
# %% 셀 3b: Stage 1 (MergedLayoutModel) 로드 — Stage 2 평가 시 pred heatmap 생성용
D_PIX = 128
STAGE1_CKPT = "./model/8_layout_1_merged/best.pt"


class MergedLayoutModel(nn.Module):
    def __init__(self, n_channels, emb_dim=EMB_DIM, d_model=D_MODEL, d_pix=D_PIX,
                 n_heads=N_HEADS, d_ff=D_FF, dropout=0.0):
        super().__init__()
        self.d_model = d_model
        self.d_pix = d_pix

        self.ch_proj = nn.Sequential(nn.Linear(emb_dim + 1, d_model // 2), nn.GELU())
        self.vid_proj = nn.Sequential(nn.Linear(emb_dim + 1, d_model // 2), nn.GELU())
        self.stt_proj = nn.Sequential(nn.Linear(emb_dim + 2, d_model // 2), nn.GELU())
        self.len_proj = nn.Sequential(nn.Linear(5, d_model // 2), nn.GELU())
        self.slot_combine = nn.Sequential(
            nn.Linear(d_model * 2, d_model), nn.GELU(),
            nn.Linear(d_model, d_model))

        inst_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, activation="gelu")
        self.inst_self_attn = nn.TransformerEncoder(
            inst_layer, num_layers=N_LAYERS_INST, enable_nested_tensor=False)

        self.channel_emb = nn.Embedding(n_channels, d_model)
        self.time_proj = nn.Linear(TIME_DIM, d_model)
        self.count_proj = nn.Sequential(
            nn.Linear(1, d_model // 4), nn.GELU(),
            nn.Linear(d_model // 4, d_model))

        self.frame_proj = nn.Sequential(
            nn.Linear(d_model, d_pix), nn.GELU(),
            nn.Linear(d_pix, d_pix))
        self.spatial_codebook = nn.Parameter(torch.randn(d_pix, GRID_H, GRID_W) * 0.02)
        self.heatmap_bias = nn.Parameter(torch.zeros(GRID_H, GRID_W))
        self.logit_scale = nn.Parameter(torch.tensor(1.0 / math.sqrt(d_pix)))

    def encode_inst(self, channel_ids, inst_diff_ch, inst_diff_vid,
                    inst_diff_stt, inst_scalars):
        ch_input = torch.cat([inst_diff_ch, inst_scalars[..., 1:2]], dim=-1)
        vid_input = torch.cat([inst_diff_vid, inst_scalars[..., 2:3]], dim=-1)
        stt_input = torch.cat([inst_diff_stt,
                               inst_scalars[..., 3:4],
                               inst_scalars[..., 4:5]], dim=-1)
        len_input = torch.cat([
            inst_scalars[..., 0:1],
            inst_scalars[..., 5:8],
            inst_scalars[..., 8:9]], dim=-1)
        proj_ch  = self.ch_proj(ch_input)
        proj_vid = self.vid_proj(vid_input)
        proj_stt = self.stt_proj(stt_input)
        proj_len = self.len_proj(len_input)
        inst_tokens = self.slot_combine(
            torch.cat([proj_ch, proj_vid, proj_stt, proj_len], dim=-1))
        ch_emb = self.channel_emb(channel_ids).unsqueeze(1)
        inst_tokens = inst_tokens + ch_emb
        inst_mask = (inst_scalars[..., 0] != 0)
        inst_pad = ~inst_mask
        inst_tokens = self.inst_self_attn(inst_tokens, src_key_padding_mask=inst_pad)
        return inst_tokens, inst_mask

    def aggregate_frames(self, inst_tokens, active_per_frame, time_feats, channel_ids):
        active_f = active_per_frame.float()
        count = active_f.sum(dim=-1, keepdim=True)
        weight = active_f / count.clamp(min=1.0)
        frame_tokens = torch.einsum('btn,bnd->btd', weight, inst_tokens)
        ch_emb = self.channel_emb(channel_ids).unsqueeze(1)
        time_emb = self.time_proj(time_feats)
        count_norm = torch.log1p(count) / math.log1p(20.0)
        count_emb = self.count_proj(count_norm)
        return frame_tokens + ch_emb + time_emb + count_emb

    def predict_heatmap(self, frame_tokens):
        proj = self.frame_proj(frame_tokens)
        heatmap = torch.einsum('btd,dhw->bthw', proj, self.spatial_codebook)
        heatmap = heatmap * self.logit_scale + self.heatmap_bias
        return heatmap

    def forward(self, channel_ids, inst_diff_ch, inst_diff_vid, inst_diff_stt,
                inst_scalars, active_per_frame, time_feats, frame_mask):
        inst_tokens, _ = self.encode_inst(
            channel_ids, inst_diff_ch, inst_diff_vid, inst_diff_stt, inst_scalars)
        frame_tokens = self.aggregate_frames(
            inst_tokens, active_per_frame, time_feats, channel_ids)
        heatmap_logits = self.predict_heatmap(frame_tokens)
        return heatmap_logits


stage1_model = MergedLayoutModel(n_channels=len(channels)).to(device)
stage1_ckpt = torch.load(STAGE1_CKPT, map_location=device, weights_only=False)
if "channel2id" in stage1_ckpt and len(stage1_ckpt["channel2id"]) != len(channel2id):
    print(f"⚠️  Stage 1 channel2id 크기 다름: {len(stage1_ckpt['channel2id'])} vs {len(channel2id)}")
stage1_model.load_state_dict(stage1_ckpt["model"])
stage1_model.eval()
for p in stage1_model.parameters():
    p.requires_grad = False
print(f"✅ Stage 1 모델 로드: {STAGE1_CKPT}")
print(f"   epoch={stage1_ckpt.get('epoch', '?')}")
print(f"   metrics={stage1_ckpt.get('metrics', '?')}")


In [ ]:
# %% 셀 4: 학습 (CenterNet — focal loss + L1/GIoU/size at positive pixels)
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

EPOCHS = 30
LR = 1e-4
WEIGHT_DECAY = 1e-2
GRAD_CLIP = 1.0

COEF_CENTER = 1.0
COEF_BBOX   = 5.0           # L1 on raw 4ch (offset_x, offset_y, log_w, log_h)
COEF_GIOU   = 2.0           # GIoU on decoded cxcywh
PEAK_NMS_RADIUS = 3
OBJ_THRESH = 0.3            # peak score threshold for inference
TOPK_PER_SEG = 50           # max peaks per segment to evaluate
W_CLAMP_MIN = 1e-3          # exp(log_w) lower clamp at decode
W_CLAMP_MAX = 1.5           # upper clamp (slight headroom over 1.0)

SAVE_DIR = "./model/8_layout_2_boxset"
os.makedirs(SAVE_DIR, exist_ok=True)

USE_BF16 = (device.type == "cuda")


def cxcywh_to_xyxy(boxes):
    cx, cy, w, h = boxes.unbind(-1)
    return torch.stack([cx - w/2, cy - h/2, cx + w/2, cy + h/2], dim=-1)


def decode_geometry_raw(raw, x_idx, y_idx, H=GRID_H, W=GRID_W):
    """raw: (..., 4) — (offset_x, offset_y, log_w_norm, log_h_norm)
    x_idx, y_idx: (...) grid coords
    Returns cxcywh in [0,1]."""
    cx = (x_idx.float() + raw[..., 0]) / W
    cy = (y_idx.float() + raw[..., 1]) / H
    w  = raw[..., 2].exp().clamp(min=W_CLAMP_MIN, max=W_CLAMP_MAX)
    h  = raw[..., 3].exp().clamp(min=W_CLAMP_MIN, max=W_CLAMP_MAX)
    return torch.stack([cx, cy, w, h], dim=-1)


def focal_loss_centernet(pred_logits, center_target, pos_mask):
    """CornerNet/CenterNet style focal loss with Gaussian negative weighting.
    pred_logits: (S, 1, H, W)
    center_target: (S, H, W) Gaussian heatmap [0,1]
    pos_mask: (S, H, W) bool — exact center pixels
    Returns: loss (scalar), n_pos (int)
    """
    pred = pred_logits.squeeze(1).sigmoid().float()
    pred_clamp = pred.clamp(1e-4, 1 - 1e-4)
    target = center_target.float()
    pos = pos_mask.float()
    neg = 1.0 - pos

    pos_loss = -((1 - pred_clamp) ** 2) * torch.log(pred_clamp) * pos
    neg_loss = -((pred_clamp ** 2) * torch.log(1 - pred_clamp)) * \
               ((1 - target) ** 4) * neg

    n_pos = pos.sum().clamp(min=1.0)
    loss = (pos_loss.sum() + neg_loss.sum()) / n_pos
    return loss, int(pos.sum().item())


def geometry_loss_at_pos(geometry, geo_target, pos_mask, seg_lengths):
    """positive pixel에서만 L1(raw 4ch) + GIoU(decoded cxcywh).
    geometry, geo_target: (S, 4, H, W) — raw (offset_x, offset_y, log_w, log_h)
    pos_mask: (S, H, W) bool
    seg_lengths: (S,) — frame-equivalent weighting
    Returns: dict of {box_loss, giou_loss, n_pos}
    """
    pos_idx = pos_mask.nonzero(as_tuple=True)        # (s_idx, y_idx, x_idx)
    n_pos = pos_idx[0].numel()
    if n_pos == 0:
        return {
            'box_loss': geometry.sum() * 0.0,
            'giou_loss': geometry.sum() * 0.0,
            'n_pos': 0,
        }

    # gather raw 4ch (n_pos, 4)
    pred = geometry[pos_idx[0], :, pos_idx[1], pos_idx[2]].float()
    gt   = geo_target[pos_idx[0], :, pos_idx[1], pos_idx[2]].float()

    seg_w = seg_lengths.float()[pos_idx[0]]                           # (n_pos,)
    w_total = seg_w.sum().clamp(min=1.0)

    # L1 on raw 4 channels — offset과 log-size 모두 한 번에 supervision
    l1 = (pred - gt).abs().sum(dim=-1)                                # (n_pos,)
    box_loss = (l1 * seg_w).sum() / w_total

    # GIoU on decoded cxcywh (작은 박스 미세조정에 도움)
    pred_cxcywh = decode_geometry_raw(pred, pos_idx[2], pos_idx[1])
    gt_cxcywh   = decode_geometry_raw(gt,   pos_idx[2], pos_idx[1])
    p_xy = cxcywh_to_xyxy(pred_cxcywh)
    g_xy = cxcywh_to_xyxy(gt_cxcywh)
    a1 = (p_xy[:, 2] - p_xy[:, 0]).clamp(min=0) * (p_xy[:, 3] - p_xy[:, 1]).clamp(min=0)
    a2 = (g_xy[:, 2] - g_xy[:, 0]).clamp(min=0) * (g_xy[:, 3] - g_xy[:, 1]).clamp(min=0)
    lt = torch.maximum(p_xy[:, :2], g_xy[:, :2])
    rb = torch.minimum(p_xy[:, 2:], g_xy[:, 2:])
    wh = (rb - lt).clamp(min=0)
    inter = wh[:, 0] * wh[:, 1]
    union = a1 + a2 - inter
    iou_p = inter / union.clamp(min=1e-7)
    lt_e = torch.minimum(p_xy[:, :2], g_xy[:, :2])
    rb_e = torch.maximum(p_xy[:, 2:], g_xy[:, 2:])
    wh_e = (rb_e - lt_e).clamp(min=0)
    enc = wh_e[:, 0] * wh_e[:, 1]
    giou_p = iou_p - (enc - union) / enc.clamp(min=1e-7)
    giou_loss = ((1 - giou_p) * seg_w).sum() / w_total

    return {
        'box_loss': box_loss, 'giou_loss': giou_loss, 'n_pos': n_pos,
    }


@torch.no_grad()
def decode_peaks(center_logits, geometry, top_k=TOPK_PER_SEG, nms_radius=PEAK_NMS_RADIUS):
    """center heatmap에서 local maxima 추출 → top_k peaks → raw geometry decode.
    Returns:
      pred_boxes:  (S, top_k, 4) cxcywh in [0,1]
      pred_scores: (S, top_k)
    """
    S, _, H, W = center_logits.shape
    pred = center_logits.sigmoid().squeeze(1).float()                  # (S, H, W)

    pool = F.max_pool2d(pred.unsqueeze(1),
                        kernel_size=2 * nms_radius + 1,
                        stride=1, padding=nms_radius).squeeze(1)
    is_peak = (pred == pool) & (pred > 0)

    masked = pred.masked_fill(~is_peak, 0.0)
    top_vals, top_idx = masked.flatten(1).topk(top_k, dim=1)            # (S, k)
    y_idx = top_idx // W
    x_idx = top_idx % W

    arange_s = torch.arange(S, device=pred.device).unsqueeze(-1).expand(-1, top_k)
    raw = geometry[arange_s, :, y_idx, x_idx].float()                   # (S, k, 4) raw
    pred_boxes = decode_geometry_raw(raw, x_idx, y_idx, H=H, W=W)        # (S, k, 4) cxcywh
    return pred_boxes, top_vals


@torch.no_grad()
def compute_metrics_centernet(center_logits, geometry,
                              gt_boxes, gt_box_mask, seg_lengths):
    """CenterNet 추론 → top_k peaks를 GT와 비교.
    metrics:
      topk_IoU: top-K peaks (K=실제 GT 수)와 GT를 geom-only Hungarian → 평균 IoU
      cls_IoU: 모든 top peaks 후보와 GT를 class+geom Hungarian → 평균 IoU
      obj_P/R/F1: peak score > OBJ_THRESH 인 것들 vs GT
    seg_length로 weighted average.
    """
    pred_boxes, pred_scores = decode_peaks(center_logits, geometry,
                                            top_k=TOPK_PER_SEG,
                                            nms_radius=PEAK_NMS_RADIUS)
    pb_np = pred_boxes.cpu().numpy()                  # (S, top_k, 4)
    ps_np = pred_scores.cpu().numpy()                 # (S, top_k)
    gb_np = gt_boxes.cpu().numpy()
    gm_np = gt_box_mask.cpu().numpy()
    seg_len_np = seg_lengths.cpu().numpy()

    S = pb_np.shape[0]
    K_per_seg = gm_np.sum(axis=1)
    valid = np.where(K_per_seg > 0)[0]

    n_pred_obj_per_seg = (ps_np > OBJ_THRESH).sum(axis=1)
    n_pred_obj_total = float((n_pred_obj_per_seg * seg_len_np).sum())
    K_total = float((K_per_seg * seg_len_np).sum())

    topk_ious = []; cls_ious = []
    topk_w = []; cls_w = []
    obj_tp = 0.0

    for s in valid:
        valid_cols = np.where(gm_np[s])[0]
        K = len(valid_cols)
        gt_s = gb_np[s, valid_cols]                     # (K, 4)

        # 1) topk = top-K (K=실제 GT 수)
        K_use = min(K, TOPK_PER_SEG)
        topk_pred = pb_np[s, :K_use]                    # (K_use, 4)

        # geom cost: L1 + GIoU
        if K_use > 0:
            box_cost = np.abs(topk_pred[:, None, :] - gt_s[None, :, :]).sum(-1)
            tp_xy = _cxcywh_to_xyxy_np(topk_pred)
            gt_xy = _cxcywh_to_xyxy_np(gt_s)
            iou_topk, giou_topk = _giou_iou_np(tp_xy, gt_xy)
            cost_geom = 5.0 * box_cost - 2.0 * giou_topk
            p_idx, g_local = linear_sum_assignment(cost_geom)
            M = len(p_idx)
            topk_ious.append(iou_topk[p_idx, g_local])
            topk_w.append(np.full(M, seg_len_np[s], dtype=np.float32))

        # 2) cls = all top peaks + class-aware
        # cost: -score + 5*L1 - 2*GIoU
        all_pred = pb_np[s]                              # (top_k, 4)
        box_cost_all = np.abs(all_pred[:, None, :] - gt_s[None, :, :]).sum(-1)
        ap_xy = _cxcywh_to_xyxy_np(all_pred)
        iou_all, giou_all = _giou_iou_np(ap_xy, gt_xy)
        cost_cls = -ps_np[s][:, None] + 5.0 * box_cost_all - 2.0 * giou_all
        p_idx_b, g_local_b = linear_sum_assignment(cost_cls)
        Mb = len(p_idx_b)
        cls_ious.append(iou_all[p_idx_b, g_local_b])
        cls_w.append(np.full(Mb, seg_len_np[s], dtype=np.float32))

        # obj_F1: matched peaks (cls Hungarian) with score > thr count as TP
        matched_obj = ps_np[s, p_idx_b] > OBJ_THRESH
        obj_tp += float((matched_obj * np.full(Mb, seg_len_np[s])).sum())

    if topk_ious:
        topk_iou_arr = np.concatenate(topk_ious)
        topk_w_arr = np.concatenate(topk_w)
        topk_iou_w = float((topk_iou_arr * topk_w_arr).sum() / max(topk_w_arr.sum(), 1))
    else:
        topk_iou_w = 0.0

    if cls_ious:
        cls_iou_arr = np.concatenate(cls_ious)
        cls_w_arr = np.concatenate(cls_w)
        cls_iou_w = float((cls_iou_arr * cls_w_arr).sum() / max(cls_w_arr.sum(), 1))
    else:
        cls_iou_w = 0.0

    obj_fn = K_total - obj_tp
    obj_fp = n_pred_obj_total - obj_tp
    return {
        'topk_IoU': topk_iou_w,
        'cls_IoU': cls_iou_w,
        'obj_P': obj_tp / max(obj_tp + obj_fp, 1.0),
        'obj_R': obj_tp / max(obj_tp + obj_fn, 1.0),
        'obj_F1': (2 * obj_tp) / max(2 * obj_tp + obj_fp + obj_fn, 1.0),
    }


def _cxcywh_to_xyxy_np(b):
    cx, cy, w, h = b[..., 0], b[..., 1], b[..., 2], b[..., 3]
    return np.stack([cx - w/2, cy - h/2, cx + w/2, cy + h/2], axis=-1)


def _giou_iou_np(b1, b2):
    a1 = (b1[:, 2] - b1[:, 0]).clip(0) * (b1[:, 3] - b1[:, 1]).clip(0)
    a2 = (b2[:, 2] - b2[:, 0]).clip(0) * (b2[:, 3] - b2[:, 1]).clip(0)
    lt = np.maximum(b1[:, None, :2], b2[None, :, :2])
    rb = np.minimum(b1[:, None, 2:], b2[None, :, 2:])
    wh = (rb - lt).clip(0)
    inter = wh[..., 0] * wh[..., 1]
    union = a1[:, None] + a2[None, :] - inter
    iou = inter / np.maximum(union, 1e-7)
    lt_e = np.minimum(b1[:, None, :2], b2[None, :, :2])
    rb_e = np.maximum(b1[:, None, 2:], b2[None, :, 2:])
    wh_e = (rb_e - lt_e).clip(0)
    enc = wh_e[..., 0] * wh_e[..., 1]
    giou = iou - (enc - union) / np.maximum(enc, 1e-7)
    return iou, giou


def to_device(batch, dev):
    return {k: (v.to(dev, non_blocking=True) if isinstance(v, torch.Tensor) else v)
            for k, v in batch.items()}


SEG_CHUNK_TRAIN = 64    # CenterNet은 dense라 더 작은 chunk
SEG_CHUNK_EVAL  = 64


def _global_pos_count(batch):
    return float(batch["seg_pos_mask"].sum().clamp(min=1).item())


def train_step(batch):
    """Stage 1 pred heatmap을 입력으로 사용. Dense forward (chunked for memory)."""
    optimizer.zero_grad(set_to_none=True)

    # Stage 1 pred heatmap (no_grad, frozen)
    s1_heatmap = get_stage1_heatmap_for_segments(batch)               # (S_total, H, W)

    # Encoder once
    with torch.amp.autocast('cuda', dtype=torch.bfloat16, enabled=USE_BF16):
        inst_tokens, _ = model.encode_inst(
            batch["channel_ids"],
            batch["inst_diff_ch"], batch["inst_diff_vid"], batch["inst_diff_stt"],
            batch["inst_scalars"])

    inst_d = inst_tokens.detach().requires_grad_(True)
    inst_grad_total = torch.zeros_like(inst_d)

    S_total = batch["seg_video_idx"].shape[0]
    n_pos_global = _global_pos_count(batch)

    loss_acc = 0.0
    center_acc = 0.0
    box_acc = 0.0
    giou_acc = 0.0
    backward_done = False

    for s_start in range(0, S_total, SEG_CHUNK_TRAIN):
        s_end = min(s_start + SEG_CHUNK_TRAIN, S_total)
        if s_end <= s_start:
            continue

        with torch.amp.autocast('cuda', dtype=torch.bfloat16, enabled=USE_BF16):
            cl, gm = model.predict_segments(
                inst_d, batch["channel_ids"],
                batch["seg_active_mask"][s_start:s_end],
                s1_heatmap[s_start:s_end],
                batch["seg_time_feats"][s_start:s_end],
                batch["seg_video_idx"][s_start:s_end])

            center_l, n_pos_chunk = focal_loss_centernet(
                cl,
                batch["seg_center_target"][s_start:s_end],
                batch["seg_pos_mask"][s_start:s_end])

            geo_d = geometry_loss_at_pos(
                gm,
                batch["seg_geo_target"][s_start:s_end],
                batch["seg_pos_mask"][s_start:s_end],
                batch["seg_length"][s_start:s_end])

        center_weight = n_pos_chunk / n_pos_global if n_pos_chunk > 0 else 0.0
        center_term = COEF_CENTER * center_l * center_weight

        geo_weight = geo_d['n_pos'] / n_pos_global if geo_d['n_pos'] > 0 else 0.0
        box_term  = COEF_BBOX * geo_d['box_loss']  * geo_weight
        giou_term = COEF_GIOU * geo_d['giou_loss'] * geo_weight

        chunk_total = center_term + box_term + giou_term
        chunk_total.backward()
        backward_done = True

        if inst_d.grad is not None:
            inst_grad_total = inst_grad_total + inst_d.grad
            inst_d.grad = None

        loss_acc   += float(chunk_total.detach())
        center_acc += float(center_term.detach())
        box_acc    += float(box_term.detach())
        giou_acc   += float(giou_term.detach())

        del cl, gm, chunk_total, center_term, box_term, giou_term

    if backward_done:
        inst_tokens.backward(inst_grad_total)
    else:
        (inst_tokens.sum() * 0.0).backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    optimizer.step()
    scheduler.step()

    return loss_acc, {
        'center': center_acc, 'box': box_acc, 'giou': giou_acc,
    }


@torch.no_grad()
def eval_step(batch, heatmap_input):
    """Eval: heatmap_input은 GT(A) 또는 Stage 1 pred(B). chunked dense forward."""
    with torch.amp.autocast('cuda', dtype=torch.bfloat16, enabled=USE_BF16):
        inst_tokens, _ = model.encode_inst(
            batch["channel_ids"],
            batch["inst_diff_ch"], batch["inst_diff_vid"], batch["inst_diff_stt"],
            batch["inst_scalars"])

    S_total = batch["seg_video_idx"].shape[0]
    n_pos_global = _global_pos_count(batch)

    loss_acc = 0.0
    metrics_acc = {'topk_IoU': 0.0, 'cls_IoU': 0.0, 'obj_P': 0.0, 'obj_R': 0.0, 'obj_F1': 0.0}
    seg_len_total = float(batch["seg_length"].float().sum().clamp(min=1).item())

    for s_start in range(0, S_total, SEG_CHUNK_EVAL):
        s_end = min(s_start + SEG_CHUNK_EVAL, S_total)
        if s_end <= s_start:
            continue

        with torch.amp.autocast('cuda', dtype=torch.bfloat16, enabled=USE_BF16):
            cl, gm = model.predict_segments(
                inst_tokens, batch["channel_ids"],
                batch["seg_active_mask"][s_start:s_end],
                heatmap_input[s_start:s_end],
                batch["seg_time_feats"][s_start:s_end],
                batch["seg_video_idx"][s_start:s_end])
            center_l, n_pos_chunk = focal_loss_centernet(
                cl,
                batch["seg_center_target"][s_start:s_end],
                batch["seg_pos_mask"][s_start:s_end])
            geo_d = geometry_loss_at_pos(
                gm,
                batch["seg_geo_target"][s_start:s_end],
                batch["seg_pos_mask"][s_start:s_end],
                batch["seg_length"][s_start:s_end])

        center_weight = n_pos_chunk / n_pos_global if n_pos_chunk > 0 else 0.0
        geo_weight    = geo_d['n_pos'] / n_pos_global if geo_d['n_pos'] > 0 else 0.0
        chunk_loss = (COEF_CENTER * center_l * center_weight
                      + COEF_BBOX * geo_d['box_loss']  * geo_weight
                      + COEF_GIOU * geo_d['giou_loss'] * geo_weight)
        loss_acc += float(chunk_loss.detach())

        m = compute_metrics_centernet(
            cl, gm,
            batch["seg_gt_boxes"][s_start:s_end],
            batch["seg_gt_box_mask"][s_start:s_end],
            batch["seg_length"][s_start:s_end])
        # weighted by chunk's seg_len fraction
        seg_len_c = float(batch["seg_length"][s_start:s_end].float().sum().item())
        w = seg_len_c / seg_len_total
        for k in metrics_acc:
            metrics_acc[k] += m[k] * w

        del cl, gm, chunk_loss

    return loss_acc, metrics_acc


@torch.no_grad()
def get_stage1_heatmap_for_segments(batch):
    """Stage 1 forward — segment 단위로 pred heatmap 생성."""
    S = batch["seg_video_idx"].shape[0]
    seg_video = batch["seg_video_idx"]
    s1_channel_ids = batch["channel_ids"][seg_video]

    s1_inst_diff_ch  = batch["inst_diff_ch"][seg_video]
    s1_inst_diff_vid = batch["inst_diff_vid"][seg_video]
    s1_inst_diff_stt = batch["inst_diff_stt"][seg_video]
    s1_inst_scalars  = batch["inst_scalars"][seg_video]

    s1_active = batch["seg_active_mask"].unsqueeze(1)
    s1_time = batch["seg_time_feats"].unsqueeze(1)
    s1_fm = torch.ones(S, 1, dtype=torch.bool, device=device)

    with torch.amp.autocast('cuda', dtype=torch.bfloat16, enabled=USE_BF16):
        s1_logits = stage1_model(
            s1_channel_ids,
            s1_inst_diff_ch, s1_inst_diff_vid, s1_inst_diff_stt,
            s1_inst_scalars,
            s1_active, s1_time, s1_fm)
    return torch.sigmoid(s1_logits.float()).squeeze(1)


optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS * max(1, len(train_loader)))

best_eval = 0.0
log_path = os.path.join(SAVE_DIR, "train_log.txt")

for epoch in range(EPOCHS):
    model.train()
    tr_loss = 0.0; tr_center = 0.0; tr_box = 0.0; tr_giou = 0.0; tr_n = 0
    pbar = tqdm(train_loader, desc=f"E{epoch+1}/{EPOCHS} train")
    for batch in pbar:
        batch = to_device(batch, device)
        loss_val, stats = train_step(batch)
        tr_loss += loss_val
        tr_center += stats['center']
        tr_box += stats['box']
        tr_giou += stats['giou']
        tr_n += 1
        pbar.set_postfix(
            loss=f"{loss_val:.4f}",
            center=f"{stats['center']:.3f}",
            box=f"{stats['box']:.3f}",
            giou=f"{stats['giou']:.3f}",
            S=f"{batch['seg_video_idx'].shape[0]}")

    model.eval()
    ev_n = 0; ev_loss_A = 0.0; ev_loss_B = 0.0
    metrics_A = {'topk_IoU': 0.0, 'cls_IoU': 0.0, 'obj_P': 0.0, 'obj_R': 0.0, 'obj_F1': 0.0}
    metrics_B = {'topk_IoU': 0.0, 'cls_IoU': 0.0, 'obj_P': 0.0, 'obj_R': 0.0, 'obj_F1': 0.0}

    for batch in tqdm(eval_loader, desc=f"E{epoch+1}/{EPOCHS} eval"):
        batch = to_device(batch, device)
        s1_heatmap = get_stage1_heatmap_for_segments(batch)
        la, mA = eval_step(batch, batch["seg_merged_mask"])
        lb, mB = eval_step(batch, s1_heatmap)
        ev_loss_A += la; ev_loss_B += lb
        for k in metrics_A:
            metrics_A[k] += mA[k]; metrics_B[k] += mB[k]
        ev_n += 1

    tr_avg = tr_loss / max(tr_n, 1)
    ev_avg_A = ev_loss_A / max(ev_n, 1); ev_avg_B = ev_loss_B / max(ev_n, 1)
    for k in metrics_A:
        metrics_A[k] /= max(ev_n, 1); metrics_B[k] /= max(ev_n, 1)

    line = (f"E{epoch+1:02d}  train={tr_avg:.4f}  "
            f"center={tr_center/max(tr_n,1):.3f} box={tr_box/max(tr_n,1):.3f} "
            f"giou={tr_giou/max(tr_n,1):.3f}\n"
            f"  [A:GT  ] eval={ev_avg_A:.4f}  topk={metrics_A['topk_IoU']:.4f}  "
            f"cls={metrics_A['cls_IoU']:.4f}  objF1={metrics_A['obj_F1']:.4f}  "
            f"P={metrics_A['obj_P']:.3f}  R={metrics_A['obj_R']:.3f}\n"
            f"  [B:S1  ] eval={ev_avg_B:.4f}  topk={metrics_B['topk_IoU']:.4f}  "
            f"cls={metrics_B['cls_IoU']:.4f}  objF1={metrics_B['obj_F1']:.4f}  "
            f"P={metrics_B['obj_P']:.3f}  R={metrics_B['obj_R']:.3f}")
    print(line)
    with open(log_path, "a") as f:
        f.write(line + "\n")

    score = metrics_B['topk_IoU']
    if score > best_eval:
        best_eval = score
        torch.save({
            "model": model.state_dict(),
            "epoch": epoch + 1,
            "metrics_A": metrics_A,
            "metrics_B": metrics_B,
            "channel2id": channel2id,
        }, os.path.join(SAVE_DIR, "best.pt"))
        print(f"  ✅ best saved (B topk_IoU={score:.4f})")

    torch.save({"model": model.state_dict(), "epoch": epoch + 1},
               os.path.join(SAVE_DIR, "last.pt"))
